# Data Preprocessing

This notebook prepares the Raisin Dataset for the two models. It applies only the required steps and learns every preprocessing value from training data.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

## 1. Load and Split the Dataset

A stratified 60/20/20 split keeps the 50/50 class ratio in training, validation, and test data.

In [ ]:
CURRENT_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name == "notebooks"
    else CURRENT_DIRECTORY
)
DATA_PATH = (
    PROJECT_ROOT / "data" / "raw" / "raisin" / "Raisin_Dataset.xlsx"
)

TARGET_COLUMN = "Class"
RANDOM_SEED = 42
TARGET_ENCODING = {"Kecimen": 0, "Besni": 1}

df = pd.read_excel(DATA_PATH)

train_df, temporary_df = train_test_split(
    df,
    test_size=0.40,
    random_state=RANDOM_SEED,
    stratify=df[TARGET_COLUMN],
)
validation_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temporary_df[TARGET_COLUMN],
)

print("Training:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

## 2. Encode the Target

`Kecimen` is encoded as `0` and `Besni` as `1`. This is only a modeling convention.

In [ ]:
def encode_target(target_series):
    encoded = target_series.map(TARGET_ENCODING)
    if encoded.isna().any():
        raise ValueError("Unexpected target class.")
    return encoded.to_numpy(dtype=np.int64)


y_train = encode_target(train_df[TARGET_COLUMN])
y_validation = encode_target(validation_df[TARGET_COLUMN])
y_test = encode_target(test_df[TARGET_COLUMN])

print("Training class counts:", np.bincount(y_train))
print("Validation class counts:", np.bincount(y_validation))
print("Test class counts:", np.bincount(y_test))

## 3. Check the Input Features

The seven inputs are numerical and contain no missing values, so imputation and one-hot encoding are not needed.

In [ ]:
feature_names = [
    column for column in df.columns
    if column != TARGET_COLUMN
]
missing_counts = df.isna().sum()

assert missing_counts.sum() == 0
assert all(pd.api.types.is_numeric_dtype(df[column]) for column in feature_names)

X_train_raw = train_df[feature_names].to_numpy(dtype=np.float64)
X_validation_raw = validation_df[feature_names].to_numpy(dtype=np.float64)
X_test_raw = test_df[feature_names].to_numpy(dtype=np.float64)

pd.DataFrame({
    "data_type": df[feature_names].dtypes.astype(str),
    "missing_count": missing_counts[feature_names],
})

## 4. Standardize the Original Features

Training means and population standard deviations are used for every split:

$$x_{scaled}=\frac{x-\mu_{train}}{\sigma_{train}}$$

In [ ]:
training_means = X_train_raw.mean(axis=0)
training_stds = X_train_raw.std(axis=0)
safe_training_stds = np.where(training_stds == 0, 1.0, training_stds)

def standardize(X, means, stds):
    return (X - means) / stds


X_train_experiment_a = standardize(
    X_train_raw, training_means, safe_training_stds
)
X_validation_experiment_a = standardize(
    X_validation_raw, training_means, safe_training_stds
)
X_test_experiment_a = standardize(
    X_test_raw, training_means, safe_training_stds
)

print("Scaled training means:", X_train_experiment_a.mean(axis=0).round(6))
print("Scaled training stds:", X_train_experiment_a.std(axis=0).round(6))

## 5. Create Degree-Two Features

Experiment B contains the seven scaled features, their squares, and every unique pairwise interaction. Both models receive the same 35 columns.

In [ ]:
def create_polynomial_features(X, names):
    columns = [X, X**2]
    polynomial_names = list(names) + [f"{name}^2" for name in names]

    interactions = []
    for first in range(X.shape[1]):
        for second in range(first + 1, X.shape[1]):
            interactions.append(X[:, first] * X[:, second])
            polynomial_names.append(f"{names[first]}*{names[second]}")

    columns.append(np.column_stack(interactions))
    return np.column_stack(columns), polynomial_names


X_train_polynomial, experiment_b_feature_names = create_polynomial_features(
    X_train_experiment_a, feature_names
)
X_validation_polynomial, _ = create_polynomial_features(
    X_validation_experiment_a, feature_names
)
X_test_polynomial, _ = create_polynomial_features(
    X_test_experiment_a, feature_names
)

polynomial_means = X_train_polynomial.mean(axis=0)
polynomial_stds = X_train_polynomial.std(axis=0)
safe_polynomial_stds = np.where(polynomial_stds == 0, 1.0, polynomial_stds)

X_train_experiment_b = standardize(
    X_train_polynomial, polynomial_means, safe_polynomial_stds
)
X_validation_experiment_b = standardize(
    X_validation_polynomial, polynomial_means, safe_polynomial_stds
)
X_test_experiment_b = standardize(
    X_test_polynomial, polynomial_means, safe_polynomial_stds
)

print("Experiment B feature count:", len(experiment_b_feature_names))

## 6. Experiment Summary

The test set remains separate. Within each experiment, linear and logistic regression receive identical matrices.

In [ ]:
experiment_summary = pd.DataFrame([
    {
        "experiment": "A: scaled original features",
        "training_shape": X_train_experiment_a.shape,
        "validation_shape": X_validation_experiment_a.shape,
        "test_shape": X_test_experiment_a.shape,
    },
    {
        "experiment": "B: degree-two features",
        "training_shape": X_train_experiment_b.shape,
        "validation_shape": X_validation_experiment_b.shape,
        "test_shape": X_test_experiment_b.shape,
    },
])

assert np.isfinite(X_train_experiment_a).all()
assert np.isfinite(X_train_experiment_b).all()
assert set(np.unique(y_train)) == {0, 1}

experiment_summary